### all imports

In [1]:
import socket
import firebase_admin
from firebase_admin import credentials, firestore
from threading import Lock, Thread
import json

# New import
from datetime import datetime

### connection|

In [2]:
class FirestoreConnection:
    _instance = None
    _lock = Lock()

    def __new__(cls, *args, **kwargs):
        if not cls._instance:
            with cls._lock:
                if not cls._instance:
                    cls._instance = super(FirestoreConnection, cls).__new__(cls)
                    cls._instance._initialize_connection(*args, **kwargs)
        return cls._instance

    def _initialize_connection(self, service_account_key: str):
        # Initialize the Firebase app with the service account if not already done
        if not firebase_admin._apps:
            cred = credentials.Certificate(service_account_key)
            firebase_admin.initialize_app(cred)
            self.db = firestore.client()
            print("Connected to Firestore")
        else:
            self.db = firestore.client()
            print("Firestore connection reused")

    def get_db(self):
        return self.db
    
    # New method to read all documents from the 'posts' collection
    def get_all_posts(self):
        posts_ref = db.collection('posts')
        try:
            # Retrieve all documents in the 'posts' collection
            docs = posts_ref.stream()

            posts_list = []
            for doc in docs:
                post_data = doc.to_dict()
                post_data['id'] = doc.id  # Add document ID to the data
                posts_list.append(post_data)

            return posts_list
        except Exception as e:
            print(f"An error occurred while reading posts: {e}")
            return None

### CRUD Design

In [ ]:
class FirestoreCRUD:
    def __init__(self):
        self.db = FirestoreConnection("file.json").get_db()  # Path to your Firebase credentials file
    

    def read_all_tuios(self):
        try:
            tuios_ref = self.db.collection("TUIO").where('isDeleted', '==', False)
            docs = tuios_ref.stream()

            tuios_list = []
            for doc in docs:
                tuio_data = doc.to_dict()
                
                # Calculate the number of posts by counting references in the 'Posts' field
                posts_count = len(tuio_data.get("Posts", []))  # Get length of 'Posts' array

                # Format Firestore timestamp to string for createdAt
                createdAt = tuio_data.get('createdAt')
                if createdAt:
                    createdAt = createdAt.isoformat() if hasattr(createdAt, 'isoformat') else str(createdAt)

                # Prepare filtered TUIO data to be sent to the client
                tuio_data_filtered = {
                    'tuio_id': doc.id,
                    'description': tuio_data.get('description', ''),
                    'createdAt': createdAt,
                    'posts_count': posts_count  # Calculated posts count
                }
                tuios_list.append(tuio_data_filtered)

            return tuios_list
        except Exception as e:
            print(f"An error occurred while reading TUIOs: {e}")
            return None


    def get_posts_by_tuio(self, tuio_id):
        try:
            # Fetch the TUIO document by its ID
            tuio_ref = self.db.collection("TUIO").document(tuio_id)
            tuio_doc = tuio_ref.get()

            if not tuio_doc.exists:
                print(f"No TUIO document found with ID {tuio_id}.")
                return None

            tuio_data = tuio_doc.to_dict()

            # Check if 'Posts' exists in the TUIO document and is not empty
            if 'Posts' in tuio_data and tuio_data['Posts']:
                posts_data = []
                for post_ref in tuio_data['Posts']:
                    post_doc = post_ref.get()
                    if post_doc.exists:
                        post_data = post_doc.to_dict()
                        post_data['post_id'] = post_ref.id  # Include the post ID
                        posts_data.append(post_data)

                return posts_data
            else:
                print(f"TUIO document {tuio_id} has no associated posts.")
                return []

        except Exception as e:
            print(f"An error occurred while retrieving posts for TUIO {tuio_id}: {e}")
            return None
        

    # Create a new post in the "Posts" collection and append it to a specified TUIO
    def create_post(self, post_data, tuio_id=None):
        print("entered")
        try:
            # Step 1: Add isDeleted field and set createdAt timestamp
            post_data['isDeleted'] = False  # Ensure isDeleted is set to False
            post_data['createdAt'] = datetime.now().isoformat()

            # Create the new post document
            post_ref = self.db.collection('Posts').document()  # Auto-generate document ID
            post_ref.set(post_data)
            new_post_id = post_ref.id

            # Step 2: Append the new post to the specified TUIO document if tuio_id is provided
            if tuio_id:
                tuio_ref = self.db.collection("TUIO").document(tuio_id)
                tuio_doc = tuio_ref.get()

                if tuio_doc.exists:
                    # Retrieve the existing posts in the TUIO document, or initialize an empty list
                    tuio_data = tuio_doc.to_dict()
                    post_refs = tuio_data.get('Posts', [])
                    
                    # Append the new post reference
                    post_refs.append(post_ref)
                    
                    # Update the TUIO document with the modified Posts array
                    tuio_ref.update({'Posts': post_refs})
                    print(f"Post {new_post_id} appended to TUIO {tuio_id}.")
                else:
                    print(f"TUIO document with ID {tuio_id} does not exist.")

            return f"Post created with ID: {new_post_id}"
        
        except Exception as e:
            return f"Error creating post: {e}"

    # Read a post by ID
    def read_post(self, post_id):
        try:
            post_ref = self.db.collection('Posts').document(post_id)
            doc = post_ref.get()
            if doc.exists:
                return json.dumps(doc.to_dict())  # Convert Firestore document to JSON
            else:
                return "Post not found."
        except Exception as e:
            return f"Error reading post: {e}"

    # Update an existing post
    def update_post(self, post_id, new_data):
        try:
            post_ref = self.db.collection('Posts').document(post_id)
            new_data['updatedAt'] = datetime.now().isoformat()
            post_ref.update(new_data)
            return f"Post {post_id} updated successfully."
        except Exception as e:
            return f"Error updating post: {e}"

    # Delete a post by ID (either soft-delete or hard-delete)
    def delete_post(self, post_id, soft_delete=True):
        try:
            post_ref = self.db.collection('Posts').document(post_id)
            if soft_delete:
                post_ref.update({"isDeleted": True, "updatedAt": datetime.now().isoformat()})
                return f"Post {post_id} marked as deleted."
            else:
                post_ref.delete()
                return f"Post {post_id} deleted successfully."
        except Exception as e:
            return f"Error deleting post: {e}"

    # CRUD for TUIO Collection (handling multiple posts)
    def create_tuio_document(self, doc_id, data, post_ids=None):
        if post_ids:
            post_refs = [self.db.collection("Posts").document(post_id) for post_id in post_ids]
            data['Posts'] = post_refs
        data['isDeleted'] = False
        self.db.collection("TUIO").document(doc_id).set(data)
        print(f"TUIO document {doc_id} created with posts: {post_ids if post_ids else []}")

    def read_tuio_document(self, tuio_id):
        doc_ref = self.db.collection("TUIO").document(tuio_id)
        doc = doc_ref.get()
        if doc.exists():
            tuio_data = doc.to_dict()
            if not tuio_data.get('isDeleted', False):
                posts_data = []
                if 'Posts' in tuio_data:
                    for post_ref in tuio_data['Posts']:
                        post_doc = post_ref.get()
                        if post_doc.exists():
                            post_data_with_id = {
                                'post_id': post_ref.id,
                                'data': post_doc.to_dict()
                            }
                            posts_data.append(post_data_with_id)
                tuio_data['Posts'] = posts_data
                return tuio_data
            else:
                print(f"TUIO document {tuio_id} is marked as deleted.")
        else:
            print(f"No TUIO document found with ID {tuio_id}.")
        return None

    def update_tuio_document(self, tuio_id, updates):
        tuio_ref = self.db.collection("TUIO").document(tuio_id)
        updates['updatedAt'] = datetime.utcnow()
        tuio_ref.update(updates)
        print(f"TUIO document {tuio_id} updated.")

    def delete_tuio_document(self, tuio_id):
        tuio_ref = self.db.collection("TUIO").document(tuio_id)
        tuio_ref.update({'isDeleted': True, 'updatedAt': datetime.utcnow()})
        print(f"TUIO {tuio_id} marked as deleted.")

        tuio_data = self.read_tuio_document(tuio_id)
        if tuio_data and 'Posts' in tuio_data:
            for post in tuio_data['Posts']:
                self.delete_post(post['post_id'], tuio_id)
    
    # CRUD Operations for User Collection
    def create_user_document(self, doc_id, data):
        try:
            # Use datetime.now() to set the createdAt and updatedAt fields
            current_time = datetime.now()
            data['createdAt'] = current_time  # Set to the current time when the document is created
            data['updatedAt'] = current_time  # Set to the current time when the document is created
            data['isDeleted'] = False  # For soft-delete logic
            
            # Create the user document
            self.db.collection("user").document(doc_id).set(data)
            return f"User {doc_id} created successfully."
        except Exception as e:
            return f"Error creating user: {e}"
    
    def read_user_document(self, doc_id):
        doc_ref = self.db.collection("user").document(doc_id)
        doc = doc_ref.get()
        if doc.exists():
            user_data = doc.to_dict()
            if not user_data.get('isDeleted', False):
                return user_data
            else:
                print(f"User {doc_id} is marked as deleted.")
        else:
            print(f"No user found with ID {doc_id}.")
        return None

    def update_user_document(self, user_id, updates):
        user_ref = self.db.collection("user").document(user_id)
        updates['updatedAt'] = datetime.utcnow()
        user_ref.update(updates)
        print(f"User {user_id} updated.")

    def delete_user_document(self, doc_id):
        user_ref = self.db.collection("user").document(doc_id)
        user_ref.update({'isDeleted': True, 'updatedAt': datetime.utcnow()})
        print(f"User {doc_id} marked as deleted.")

        posts_ref = self.db.collection("Posts").where('user_id', '==', doc_id).where('isDeleted', '==', False).stream()
        for post in posts_ref:
            self.delete_post(post.id)
    
    def read_all_users(self):
        try:
            # Fetch all documents from the 'user' collection where isDeleted is False
            users_ref = self.db.collection("user").where('isDeleted', '==', False)
            docs = users_ref.stream()

            user_list = []
            for doc in docs:
                user_data = doc.to_dict()
                user_data['user_id'] = doc.id  # Add document ID as 'user_id'
                
                # Convert non-serializable fields (e.g., Firestore timestamp) to ISO format
                user_data = self._convert_to_serializable(user_data)
                
                user_list.append(user_data)

            return user_list
        except Exception as e:
            print(f"An error occurred while reading users: {e}")
            return None

    
    def read_all_posts(self):
        try:
            # Fetch all documents from the 'Posts' collection where isDeleted is False
            posts_ref = self.db.collection('Posts').where('isDeleted', '==', False)
            docs = posts_ref.stream()

            posts_list = []
            for doc in docs:
                post_data = doc.to_dict()
                post_data['post_id'] = doc.id  # Add document ID to the data
                posts_list.append(post_data)

            return posts_list
        except Exception as e:
            print(f"An error occurred while reading posts: {e}")
            return None

    def _convert_to_serializable(self, data):
        for key, value in data.items():
            # Check if the value is a datetime-like object (including DatetimeWithNanoseconds)
            if hasattr(value, "isoformat"):
                data[key] = value.isoformat()  # Convert datetime to ISO format
            elif isinstance(value, firestore.DocumentReference):
                data[key] = value.id  # Convert DocumentReference to document ID
            elif isinstance(value, bytes):
                data[key] = value.decode('utf-8')  # Decode bytes to a UTF-8 string
            # Add more type checks as necessary
        return data


    def list_collections_and_documents(self):
        try:
            # Get all top-level collections
            collections = self.db.collections()
            
            print("Collections and their contents in the Firestore database:")

            for collection in collections:
                print(f"\nCollection: {collection.id}")
                
                # Get all documents in the collection
                docs = collection.stream()
                
                for doc in docs:
                    print(f"  Document ID: {doc.id}")
                    print(f"  Data: {doc.to_dict()}")
                    
        except Exception as e:
            print(f"Error retrieving collections or documents: {e}")    

### constructing sockets

In [4]:
# Instantiate the CRUD object
crud = FirestoreCRUD()

# Server details
server_ip = '192.168.1.17'  # Replace with your IP address
port = 8001

# Create a socket object
soc = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# Bind the socket to the IP and port
soc.bind((server_ip, port))

# Enable the server to accept connections (backlog set to 5)
soc.listen(5)

print(f"Server is running on {server_ip}:{port} and waiting for connections...")

Connected to Firestore
Server is running on 192.168.1.17:8001 and waiting for connections...


### Handeling Client

In [ ]:
def handle_client(client_socket):
    try:
        # Receive data from the client
        # data = client_socket.recv(1024).decode('ascii') changed
        data = client_socket.recv(1024).decode('utf-8')
        response = "empty"  # Default response for unknown operations
        # Process received data (assuming it's a JSON string with 'operation' and 'data')
        if data:
            request = json.loads(data)
            operation = request.get('operation')
            content = request.get('data')

            match operation:
                case 'create_post':
                    
                    post_data = content.get('post_data')
                    post_data = {'content': post_data}
                    tuio_id = str(content.get('tuio_id'))  # Retrieve TUIO ID if provided
                    print(post_data)
                    print(tuio_id)
                    response = crud.create_post(post_data, tuio_id=tuio_id)
            
                case 'read_post':
                    post_id = content.get('post_id')
                    response = crud.read_post(post_id)
            
                case 'delete_post':
                    post_id = content.get('post_id')
                    response = crud.delete_post(post_id)
            
                case 'update_post':
                    post_id = content.get('post_id')
                    response = crud.update_post(post_id)
            
                case 'create_tuio':
                    tuio_id = content.get('tuio_id')
                    tuio_data = content.get('data')
                    post_ids = content.get('post_ids', [])
                    response = crud.create_tuio_document(tuio_id, tuio_data, post_ids)
            
                case 'read_tuio':
                    tuio_id = content.get('tuio_id')
                    response = crud.read_tuio_document(tuio_id)
            
                case 'update_tuio':
                    tuio_id = content.get('tuio_id')
                    updates = content.get('updates', {})
                    response = crud.update_tuio_document(tuio_id, updates)
            
                case 'delete_tuio':
                    tuio_id = content.get('tuio_id')
                    response = crud.delete_tuio_document(tuio_id)

                case 'read_all_tuios':
                    tuios = crud.read_all_tuios()
                    response = json.dumps(tuios, ensure_ascii=False)
            
                case 'create_user':
                    user_id = content.get('user_id')
                    user_data = content.get('data')
                    response = crud.create_user_document(user_id, user_data)
            
                case 'read_user':
                    user_id = content.get('user_id')
                    response = crud.read_user_document(user_id)
            
                case 'update_user':
                    user_id = content.get('user_id')
                    updates = content.get('updates', {})
                    response = crud.update_user_document(user_id, updates)
            
                case 'delete_user':
                    user_id = content.get('user_id')
                    response = crud.delete_user_document(user_id)
            
                case 'read_all_users':
                    users = crud.read_all_users()
                    response = users if isinstance(users, str) else json.dumps(users, ensure_ascii=False)
            
                case 'read_all_posts':
                    posts = crud.read_all_posts()
                    response = posts if isinstance(posts, str) else json.dumps(posts, ensure_ascii=False)

                case 'get_posts_by_tuio':
                    tuio_id = content.get('tuio_id')
                    posts = crud.get_posts_by_tuio(tuio_id)
                    response = {"posts": posts}
                    client_socket.send(json.dumps(response).encode('utf-8'))

                case _:
                    response = "Unknown operation requested."

            if operation == 'get_posts_by_tuio':
               client_socket.send(response.encode('utf-8'))

            

    except Exception as e:
        error_message = f"Error handling client request: {e}"
        print(error_message)
        client_socket.send(error_message.encode('utf-8'))

    finally:
        # Close the connection
        client_socket.close()

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 93)

### new handle client handle

In [ ]:
while True:
    # Wait for a connection
    client_socket, client_address = soc.accept()
    print(f"Connected to client at {client_address}")

    # Handle client request in a separate thread
    client_handler = Thread(target=handle_client, args=(client_socket,))
    client_handler.start()

Connected to client at ('192.168.1.17', 63821)
Error handling client request: 'dict' object has no attribute 'encode'


## sample Creation for Haidy (posts & tuios)

In [ ]:
# Step 1: Create posts and collect their IDs for TUIO3
post_ids_tuio3 = []

# Create posts with concise content for TUIO3 and store their IDs
post_ref = crud.db.collection('Posts').document()  # Auto-generate document ID
post_data = {'content': 'hello from tuio 5!', 'createdAt': datetime.now().isoformat()}
post_ref.set(post_data)
post_ids_tuio3.append(post_ref.id)

post_ref = crud.db.collection('Posts').document()
post_data = {'content': 'once again hello from tuio 5', 'createdAt': datetime.now().isoformat()}
post_ref.set(post_data)
post_ids_tuio3.append(post_ref.id)

post_ref = crud.db.collection('Posts').document()
post_data = {'content': 'last hello from tuio 5!', 'createdAt': datetime.now().isoformat()}
post_ref.set(post_data)
post_ids_tuio3.append(post_ref.id)

# Step 2: Assign these posts to the new TUIO document (e.g., "TUIO3")
tuio_id_3 = "5"
tuio_data_3 = {'description': "this is the fifth tuio description", 'isDeleted': False}
post_refs_tuio3 = [crud.db.collection("Posts").document(post_id) for post_id in post_ids_tuio3]
tuio_data_3['Posts'] = post_refs_tuio3  # Attach post references

# Create the TUIO document for TUIO3 with references to the posts
crud.db.collection("TUIO").document(tuio_id_3).set(tuio_data_3)

print(f"TUIO document {tuio_id_3} created with linked posts: {post_ids_tuio3}")

NameError: name 'crud' is not defined

In [ ]:
# Step 1: Define the TUIO document ID
tuio_id_3 = "1"  # The TUIO document ID where posts will be linked

# Step 2: Create posts and automatically link them to TUIO3
post_data_1 = {'content': 'aaaasssssa'}
response_1 = crud.create_post(post_data_1, tuio_id=tuio_id_3)
print(response_1)

# post_data_2 = {'content': 'once again hello from tuio named tuio1'}
# response_2 = crud.create_post(post_data_2, tuio_id=tuio_id_3)
# print(response_2)

# post_data_3 = {'content': 'last hello from tuio named tuio1!'}
# response_3 = crud.create_post(post_data_3, tuio_id=tuio_id_3)
# print(response_3)

# # Step 3: Create the TUIO document if it doesn't exist
# # Only create the TUIO document if it hasn't been created yet
# tuio_data_3 = {'description': "this is the fifth tuio description", 'isDeleted': False}
# crud.db.collection("TUIO").document(tuio_id_3).set(tuio_data_3, merge=True)  # Use merge=True to avoid overwriting linked posts

# print(f"TUIO document {tuio_id_3} updated with linked posts.")

entered
Post YZ5B05giXs1caXo4YpDg appended to TUIO 1.
Post created with ID: YZ5B05giXs1caXo4YpDg


## **Open MediaPipe**

### **Import Liberaries**

In [1]:
import cv2
import pickle
import mediapipe as mp
from dollarpy import Recognizer

### **Prepare Parameters**

In [ ]:
mp_drawing = mp.solutions.drawing_utils # Drawing helpers
mp_drawing_styles = mp.solutions.drawing_styles
mp_holistic = mp.solutions.holistic # Mediapipe Solutions

### **Load Recognizer**

In [ ]:
# Load the recognizer from the file
with open('recognizer_model.pkl', 'rb') as f:
    recognizer = pickle.load(f)

### **Open Camera & Test**

In [ ]:
def getRightHandPoints():
    # Initialize lists to store points with unique stroke IDs
    right_wrist, right_thumb_cmc, right_thumb_mcp, right_thumb_ip, right_thumb_tip = (
        [],
        [],
        [],
        [],
        [],
    )
    right_index_mcp, right_index_pip, right_index_dip, right_index_tip = [], [], [], []
    right_middle_mcp, right_middle_pip, right_middle_dip, right_middle_tip = (
        [],
        [],
        [],
        [],
    )
    right_ring_mcp, right_ring_pip, right_ring_dip, right_ring_tip = [], [], [], []
    right_pinky_mcp, right_pinky_pip, right_pinky_dip, right_pinky_tip = [], [], [], []

    cap = cv2.VideoCapture(0)
    with mp_holistic.Holistic(
        min_detection_confidence=0.5, min_tracking_confidence=0.5
    ) as holistic:
        frameCount = [0]

        while True:
            ret, frame = cap.read()

            if ret:
                image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image.flags.writeable = False
                results = holistic.process(image)
                try:
                    if results.right_hand_landmarks:
                        frameCount[0] += 1
                        right_wrist.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.WRIST
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.WRIST
                                ].y,
                                1,
                            )
                        )
                        right_thumb_cmc.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_CMC
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_CMC
                                ].y,
                                2,
                            )
                        )
                        right_thumb_mcp.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_MCP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_MCP
                                ].y,
                                3,
                            )
                        )
                        right_thumb_ip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_IP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_IP
                                ].y,
                                4,
                            )
                        )
                        right_thumb_tip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_TIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.THUMB_TIP
                                ].y,
                                5,
                            )
                        )
                        right_index_mcp.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_MCP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_MCP
                                ].y,
                                6,
                            )
                        )
                        right_index_pip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_PIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_PIP
                                ].y,
                                7,
                            )
                        )
                        right_index_dip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_DIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_DIP
                                ].y,
                                8,
                            )
                        )
                        right_index_tip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_TIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.INDEX_FINGER_TIP
                                ].y,
                                9,
                            )
                        )
                        right_middle_mcp.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_MCP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_MCP
                                ].y,
                                10,
                            )
                        )
                        right_middle_pip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_PIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_PIP
                                ].y,
                                11,
                            )
                        )
                        right_middle_dip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_DIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_DIP
                                ].y,
                                12,
                            )
                        )
                        right_middle_tip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_TIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.MIDDLE_FINGER_TIP
                                ].y,
                                13,
                            )
                        )
                        right_ring_mcp.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_MCP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_MCP
                                ].y,
                                14,
                            )
                        )
                        right_ring_pip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_PIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_PIP
                                ].y,
                                15,
                            )
                        )
                        right_ring_dip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_DIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_DIP
                                ].y,
                                16,
                            )
                        )
                        right_ring_tip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_TIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.RING_FINGER_TIP
                                ].y,
                                17,
                            )
                        )
                        right_pinky_mcp.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_MCP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_MCP
                                ].y,
                                18,
                            )
                        )
                        right_pinky_pip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_PIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_PIP
                                ].y,
                                19,
                            )
                        )
                        right_pinky_dip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_DIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_DIP
                                ].y,
                                20,
                            )
                        )
                        right_pinky_tip.append(
                            Point(
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_TIP
                                ].x,
                                results.right_hand_landmarks.landmark[
                                    mp_holistic.HandLandmark.PINKY_TIP
                                ].y,
                                21,
                            )
                        )
                        if frameCount[0] >= 30:
                            print("frame count reached")
                            frameCount[0] = 0
                            points = (
                                right_wrist
                                + right_thumb_cmc
                                + right_thumb_mcp
                                + right_thumb_ip
                                + right_thumb_tip
                                + right_index_mcp
                                + right_index_pip
                                + right_index_dip
                                + right_index_tip
                                + right_middle_mcp
                                + right_middle_pip
                                + right_middle_dip
                                + right_middle_tip
                                + right_ring_mcp
                                + right_ring_pip
                                + right_ring_dip
                                + right_ring_tip
                                + right_pinky_mcp
                                + right_pinky_pip
                                + right_pinky_dip
                                + right_pinky_tip
                            )
                            predection = recognizer.recognize(points)
                            print(predection[0])
                            print("HEllo")
                            cv2.putText(
                                frame,
                                str(predection[0]),
                                (10, 30),  # Position on the screen (top left)
                                cv2.FONT_HERSHEY_SIMPLEX,
                                1,  # Font size
                                (255, 0, 0),  # Font color (blue in BGR)
                                2,  # Thickness of the text
                                cv2.LINE_AA,
                            )
                except:
                    pass
                image.flags.writeable = True
                image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
                mp_drawing.draw_landmarks(
                    image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS
                )
            else:
                cap.release()
                cv2.destroyAllWindows()
                cv2.waitKey(100)
                break

            if results.right_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS
                )

            # Display the frame with drawn text and landmarks
            cv2.imshow("Hand Tracking", frame)

            # Press 'q' to break out of the loop
            if cv2.waitKey(10) & 0xFF == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()

In [ ]:
getRightHandPoints()